# Sequence and Variant Generation with GOOSE

This notebook demonstrates the user-facing API in `goose.create`. It walks through every public function:

**Sequence generation**
- `goose.sequence()` -- generate an IDR by specifying physicochemical properties (FCR, NCPR, hydropathy, kappa).
- `goose.seq_by_fractions()` -- generate an IDR by specifying per-amino-acid fractions.
- `goose.seq_by_classes()` -- generate an IDR by specifying amino-acid *class* fractions (aromatic, polar, positive, etc.).
- `goose.seq_by_re()` -- generate an IDR with a target end-to-end distance (Re).
- `goose.seq_by_rg()` -- generate an IDR with a target radius of gyration (Rg).

**Variant generation**
- `goose.variant()` -- generate variants of an existing IDR using one of 16 supported variant types.

For each function we will:
1. Call the function with a representative set of arguments.
2. Use [SPARROW](https://github.com/idptools/sparrow) to confirm the resulting sequence has the requested properties.

In [2]:
import goose
from sparrow import Protein

# A small helper to print a few sequence properties consistently.
def describe(seq, label=''):
    p = Protein(seq)
    print(f"{label}")
    print(f"  length:     {len(seq)}")
    print(f"  FCR:        {p.FCR:.3f}")
    print(f"  NCPR:       {p.NCPR:.3f}")
    print(f"  hydropathy: {p.hydrophobicity:.3f}")
    print(f"  kappa:      {p.kappa:.3f}")
    print(f"  sequence:   {seq}")

## 1. `goose.sequence()` -- generate by properties

`sequence()` is the main entry point for building an IDR with a specific combination of physicochemical properties. You can specify any subset of:
- `FCR` (fraction of charged residues, 0-1)
- `NCPR` (net charge per residue, -1 to 1)
- `hydropathy` (mean Kyte-Doolittle hydropathy, 0-6.6)
- `kappa` (charge patterning, 0-1)

In [3]:
# Build a 100-residue IDR with specified FCR, NCPR, and hydropathy.
seq = goose.sequence(100, FCR=0.30, NCPR=0.10, hydropathy=3.0)
describe(seq, label='sequence() with FCR/NCPR/hydropathy')

sequence() with FCR/NCPR/hydropathy
  length:     100
  FCR:        0.300
  NCPR:       0.100
  hydropathy: 2.977
  kappa:      0.180
  sequence:   GRQKEYTKIEAYDPPHKTMCTNLNTQRGRKKDFNKLKHSLQILGPNPKGKIGFGGSKPGSKHNIRPQSGKSHGGDNHTEEVERDMSPRDQNFPLMKNNRP


In [4]:
# You can also specify kappa to control how the +/- charges are distributed.
# Low kappa -> well-mixed charges; high kappa -> blocky/segregated charges.
seq_low_kappa  = goose.sequence(120, FCR=0.30, NCPR=0.0, kappa=0.10)
seq_high_kappa = goose.sequence(120, FCR=0.30, NCPR=0.0, kappa=0.50)
describe(seq_low_kappa,  label='Low kappa (well mixed +/-)')
print()
describe(seq_high_kappa, label='High kappa (blocky +/-)')

Low kappa (well mixed +/-)
  length:     120
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 3.518
  kappa:      0.129
  sequence:   CSDKNIKFEDDEGGKIGFRFRPHNLGFKESAPTNSSMSKCEWSSGNSCMRDDRAPSKDPVSGDNKPVYLDQRTVDTSEQSVEPKMLKERWCLSYGSQPTELGVKAKTSEKSSHQCVSIMS

High kappa (blocky +/-)
  length:     120
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 3.604
  kappa:      0.510
  sequence:   EWDPTDDQDEGSPGPTYDLTFTSDDYEEEEVVVSAPVENDDMLIEHDITIPYLISLAGTVLQSQPPQSNGGRPIGVISLNKKIVIVNKPGARSTRRPKRMKSRRYSPSKRHKTSKAINKR


In [5]:
# Exclude specific residues from generation (e.g. cysteine and methionine).
# NOTE: you cannot exclude charged residues when FCR/NCPR are specified.
seq_no_cm = goose.sequence(80, hydropathy=2.8, exclude=['C', 'M'])
describe(seq_no_cm, label='sequence() excluding C and M')
print(f"  contains C: {'C' in seq_no_cm}")
print(f"  contains M: {'M' in seq_no_cm}")

sequence() excluding C and M
  length:     80
  FCR:        0.500
  NCPR:       -0.400
  hydropathy: 2.737
  kappa:      0.281
  sequence:   GDHRPEPEESRSPVLDPEQDDNVNDQTEDAESGFETSTEIVELTSDSEATDIVYDEAEDDNEEKKDETGEGEDDEDSDED
  contains C: False
  contains M: False


In [6]:
# Use organism-specific IDR amino-acid frequencies. Built-in options are:
# 'human', 'mouse', 'fly', 'yeast', 'neurospora', 'arabidopsis', 'e_coli',
# 'worm', 'zebrafish', 'frog', 'dictyostelium', 'unbiased', 'all'
seq_human = goose.sequence(100, FCR=0.20, hydropathy=3.0, custom_probabilities='human')
describe(seq_human, label="sequence() with custom_probabilities='human'")

sequence() with custom_probabilities='human'
  length:     100
  FCR:        0.200
  NCPR:       -0.140
  hydropathy: 2.935
  kappa:      0.202
  sequence:   TPQLTHSHKGGYSSPDLPDPDPDSHGNVTQEHKYDGQPDDNQRWAFDNEYNNSPSDNTTEDPQHNHGTYASGASDSNDSLQYQGLVSPANSEVNQNSGDF


## 2. `goose.seq_by_fractions()` -- generate by per-amino-acid fractions

Use `seq_by_fractions()` to pin the fraction of one or more specific amino acids. Any amino acids you do not list are filled in to satisfy disorder and length requirements.

In [7]:
# A 100-residue IDR that is 30% alanine and 10% glycine.
seq = goose.seq_by_fractions(100, A=0.30, G=0.10)
print(f"Alanine fraction: {seq.count('A') / len(seq):.3f} (target 0.30)")
print(f"Glycine fraction: {seq.count('G') / len(seq):.3f} (target 0.10)")
describe(seq, label='seq_by_fractions(A=0.30, G=0.10)')

Alanine fraction: 0.300 (target 0.30)
Glycine fraction: 0.100 (target 0.10)
seq_by_fractions(A=0.30, G=0.10)
  length:     100
  FCR:        0.170
  NCPR:       0.010
  hydropathy: 4.367
  kappa:      0.254
  sequence:   LAKSAQYNNRSASAAASIAEAVAKAIAGVDAGPVPGITTGAAAATQASRRKPTEFDRARAGPAQAESVGAFASEEATLSATVVAAHIGNRANGAAGGNDQ


In [8]:
# Override GOOSE's default max fraction for a given amino acid if you want
# to push it past the built-in limit.
seq = goose.seq_by_fractions(60, A=0.45, max_aa_fractions={'A': 0.50})
print(f"Alanine fraction: {seq.count('A') / len(seq):.3f} (target 0.45)")
describe(seq, label='seq_by_fractions(A=0.45, max_aa_fractions={A: 0.50})')

Alanine fraction: 0.450 (target 0.45)
seq_by_fractions(A=0.45, max_aa_fractions={A: 0.50})
  length:     60
  FCR:        0.100
  NCPR:       0.033
  hydropathy: 5.022
  kappa:      0.435
  sequence:   AAKAAHNAANTIATAAAAESDNAKLKLAAASAYAALGSAALRGSINVAAAPAAPSPAAFV


## 3. `goose.seq_by_classes()` -- generate by amino-acid class fractions

Instead of specifying individual amino-acid fractions, you can specify fractions of broader amino-acid *classes*. The recognized classes are:

| Class | Residues |
| --- | --- |
| aromatic | F, W, Y |
| aliphatic | A, I, L, V |
| polar | N, Q, S, T |
| positive | K, R |
| negative | D, E |
| glycine | G |
| proline | P |
| cysteine | C |
| histidine | H |

In [9]:
seq = goose.seq_by_classes(100, aromatic=0.10, polar=0.30, positive=0.10, negative=0.10)

frac = lambda residues: sum(seq.count(a) for a in residues) / len(seq)
print(f"aromatic (FWY): {frac('FWY'):.3f} (target 0.10)")
print(f"polar (NQST):   {frac('NQST'):.3f} (target 0.30)")
print(f"positive (KR):  {frac('KR'):.3f} (target 0.10)")
print(f"negative (DE):  {frac('DE'):.3f} (target 0.10)")
describe(seq, label='seq_by_classes(...)')

aromatic (FWY): 0.100 (target 0.10)
polar (NQST):   0.300 (target 0.30)
positive (KR):  0.100 (target 0.10)
negative (DE):  0.100 (target 0.10)
seq_by_classes(...)
  length:     100
  FCR:        0.200
  NCPR:       0.000
  hydropathy: 3.000
  kappa:      0.214
  sequence:   YSEHSPGAYSNPPWTLSGEHDTTCANNAHTQNFPPYHQTIHYQECRHHERKRHVKCCENQSGSCKQNPQNGEFNCPKQDSDWQVWEGCPHKSSMGRGYGK


## 4. `goose.seq_by_re()` -- generate with a target end-to-end distance (Re)

Specify a target end-to-end distance in Angstroms. GOOSE will iterate until it finds a disordered sequence whose predicted Re is within `allowed_error` of the target.

Achievable Re values depend on length; if you request something outside the feasible range you'll get a `GooseInputError` telling you the allowed range.

In [10]:
seq = goose.seq_by_re(80, objective_re=60.0)
p = Protein(seq)
print(f"Predicted Re: {p.predictor.end_to_end_distance():.2f}  (target 60.0)")
describe(seq, label='seq_by_re(80, 60.0)')

Predicted Re: 59.73  (target 60.0)
seq_by_re(80, 60.0)
  length:     80
  FCR:        0.263
  NCPR:       -0.038
  hydropathy: 3.409
  kappa:      0.180
  sequence:   TADGNFPLSCGNTSEASRLQHEYLSGMENSDTRWEHTPPELRSNKHPVSDDFKESTFQILREETSVTSSRRVKWSGSTTT


## 5. `goose.seq_by_rg()` -- generate with a target radius of gyration (Rg)

In [11]:
seq = goose.seq_by_rg(80, objective_rg=22.0)
p = Protein(seq)
print(f"Predicted Rg: {p.predictor.radius_of_gyration():.2f}  (target 22.0)")
describe(seq, label='seq_by_rg(80, 22.0)')

Predicted Rg: 22.37  (target 22.0)
seq_by_rg(80, 22.0)
  length:     80
  FCR:        0.275
  NCPR:       0.050
  hydropathy: 3.639
  kappa:      0.178
  sequence:   LGGPPTPLELRPGKNPSLLTLKSMRIGLARAKREPQWWWSSWFRTFFQEWDLDSFRGNDSSANIETRSDSRVRQLKDNSV


## 6. `goose.variant()` -- generate sequence variants

`variant()` is a unified dispatcher for 16 different variant types. The first argument is the starting sequence and the second is the variant type name. Additional keyword arguments are specific to the requested variant.

Below are several representative examples. See the `goose.variant` docstring for the full list.

In [12]:
# Start from a freshly designed IDR so we have a known baseline to compare against.
parent = goose.sequence(80, FCR=0.30, NCPR=0.0, hydropathy=2.8, kappa=0.15)
describe(parent, label='Parent sequence')

Parent sequence
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 2.799
  kappa:      0.129
  sequence:   KNQHELQTSDREGTDRINYWFSEEQLVKPELQQNQNDDNCSSKSSQLSFPMTPSRDKKRKDQVSVPALQSHRSENNHKQH


### 6a. Shuffle specific regions

`shuffle_specific_regions` reshuffles the residues inside the regions you pass and leaves everything else alone. Regions use 0-based, end-exclusive indexing (Python slice convention).

In [13]:
shuffled = goose.variant(parent, 'shuffle_specific_regions',
                         shuffle_regions=[(0, 20), (40, 60)])
print('parent  :', parent)
print('shuffled:', shuffled)
describe(shuffled, label='shuffle_specific_regions')

parent  : KNQHELQTSDREGTDRINYWFSEEQLVKPELQQNQNDDNCSSKSSQLSFPMTPSRDKKRKDQVSVPALQSHRSENNHKQH
shuffled: NSEDDLWQTKNQIYGRHTERFSEEQLVKPELQQNQNDDNCSPTKSPSRSQDSFRKMKLKSDQVSVPALQSHRSENNHKQH
shuffle_specific_regions
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 2.799
  kappa:      0.233
  sequence:   NSEDDLWQTKNQIYGRHTERFSEEQLVKPELQQNQNDDNCSPTKSPSRSQDSFRKMKLKSDQVSVPALQSHRSENNHKQH


### 6b. Shuffle only specific residue types

`shuffle_specific_residues` shuffles only the positions of the listed residue types. Useful for, e.g., redistributing charges while keeping every other residue exactly in place.

In [14]:
shuffled_charges = goose.variant(parent, 'shuffle_specific_residues',
                                 target_residues=['K', 'R', 'D', 'E'])
describe(shuffled_charges, label='shuffle_specific_residues (K, R, D, E)')

shuffle_specific_residues (K, R, D, E)
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 2.799
  kappa:      0.216
  sequence:   KNQHKLQTSDDEGTREINYWFSEKQLVKPRLQQNQNKRNCSSKSSQLSFPMTPSDDDKEERQVSVPALQSHRSDNNHEQH


### 6c. Constant-properties variant

`constant_properties` produces a new sequence that preserves the parent's FCR, NCPR, hydropathy, and kappa, but uses a different residue composition.

In [15]:
constant = goose.variant(parent, 'constant_properties', num_attempts=50)
describe(constant, label='constant_properties variant')
print('\nparent  :', parent)
print('variant :', constant)

constant_properties variant
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 2.824
  kappa:      0.134
  sequence:   HGYNLDENTKSHVWSRSSSECDTGPLPDGQQNSKREEMWKKDWTLLEMKADSQNGGRFQSDPRTNRNGQKQSTQTKVDPT

parent  : KNQHELQTSDREGTDRINYWFSEEQLVKPELQQNQNDDNCSSKSSQLSFPMTPSRDKKRKDQVSVPALQSHRSENNHKQH
variant : HGYNLDENTKSHVWSRSSSECDTGPLPDGQQNSKREEMWKKDWTLLEMKADSQNGGRFQSDPRTNRNGQKQSTQTKVDPT


### 6d. Change hydropathy while keeping the amino-acid class composition constant

`change_hydropathy_constant_class` swaps residues within their amino-acid class (e.g. aliphatic <-> aliphatic) so the overall composition by class is preserved while the mean hydropathy moves toward `target_hydropathy`.

In [16]:
more_hydrophobic = goose.variant(parent, 'change_hydropathy_constant_class',
                                 target_hydropathy=3.5)
describe(more_hydrophobic, label='change_hydropathy_constant_class -> 3.5')

change_hydropathy_constant_class -> 3.5
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 3.534
  kappa:      0.129
  sequence:   KTSHELQTSDKEGTDKISFFFTEETIVKPEITSTTNDDTCTTKSTTVSFPITPTKDKKKKDTISIPAITTHKTETTHKSH


### 6e. Change kappa (charge patterning)

`change_kappa` rearranges existing charged residues to hit a target kappa. FCR and NCPR are preserved because no residues are added or removed -- only repositioned.

In [17]:
patterned = goose.variant(parent, 'change_kappa', target_kappa=0.55)
describe(patterned, label='change_kappa -> 0.55')

change_kappa -> 0.55
  length:     80
  FCR:        0.300
  NCPR:       0.000
  hydropathy: 2.799
  kappa:      0.571
  sequence:   ENQHDLQETDSEGTEINYWFESDDDDEQLVPLQQNQNNCSSSSQLSKFPMTPSQRVRKKRKSVPAKLQSHRKSNNHQRHK


### 6f. Change any combination of properties

`change_any_properties` lets you pass any subset of `target_FCR`, `target_NCPR`, `target_hydropathy`, `target_kappa` to retarget those properties simultaneously. Properties you do not pass are left to vary freely.

In [18]:
retargeted = goose.variant(parent, 'change_any_properties',
                           target_FCR=0.25,
                           target_NCPR=-0.10,
                           target_hydropathy=2.5)
describe(retargeted, label='change_any_properties (FCR=0.25, NCPR=-0.10, hydro=2.5)')

change_any_properties (FCR=0.25, NCPR=-0.10, hydro=2.5)
  length:     80
  FCR:        0.250
  NCPR:       -0.100
  hydropathy: 2.438
  kappa:      0.141
  sequence:   SQQHEANSNDREGNNKASYYYTEQVAPEAEQSTQNSEQCNQQKDSSAQWPMSPQEDERDRQQANLPAANNHDNENSHRQH


### 6g. Change dimensions (Rg or Re)

`change_dimensions` produces a variant of the parent whose predicted Rg or Re moves up or down. Specify direction via `increase_or_decrease='increase'` or `'decrease'` and which dimension via `rg_or_re='rg'` or `'re'`.

In [19]:
expanded = goose.variant(parent, 'change_dimensions',
                        increase_or_decrease='increase',
                        rg_or_re='rg')

p_parent = Protein(parent)
p_expanded = Protein(expanded)
print(f"parent   Rg: {p_parent.predictor.radius_of_gyration():.2f}")
print(f"variant  Rg: {p_expanded.predictor.radius_of_gyration():.2f}  (asked: increase)")
describe(expanded, label='change_dimensions (increase Rg)')

parent   Rg: 25.19
variant  Rg: 25.84  (asked: increase)
change_dimensions (increase Rg)
  length:     80
  FCR:        0.287
  NCPR:       -0.012
  hydropathy: 2.763
  kappa:      0.120
  sequence:   KNQHELQTSDREGTDRINYWFSEEQLVKPELQQNQNDDNCSSKSSQLSFPMTPSRDKKPKDQVSPPALQSHRSENNHKQH


## Next steps

- See `demos/sequence_optimization.ipynb` for using `SequenceOptimizer` to build sequences against arbitrary user-defined property functions.
- See `demos/generate_sequences_by_interaction.ipynb` and `demos/epsilon_matrix_variants.ipynb` for designing sequences that interact with a target protein via epsilon-based properties.
- See `demos/linear_profiles.ipynb` for designing sequences to match linear profiles of properties along the sequence.